# Netflix Business Case Study

## Business Problem

Analyze the Netflix catalog data and generate insights that can help Netflix decide which type of shows or movies to produce and how the business can grow in different countries.

The analysis focuses on:
- Content mix: Movies vs TV Shows
- Country-wise content availability
- Release-year and date-added trends
- Ratings, genres, actors, and directors
- Missing values and outliers
- Business insights and recommendations

## 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('ggplot')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

In [ ]:
df = pd.read_csv('netflix.txt')
df.head()

## 2. Basic Metrics and Data Overview

In [ ]:
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nData types:')
print(df.dtypes)

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

### Initial Observations

- The dataset has 8,807 rows and 12 columns.
- Most columns are categorical/text columns.
- `release_year` is numeric.
- `date_added` should be converted to datetime.
- `type` and `rating` can be converted to category because they have repeated categorical values.
- Columns such as `director`, `cast`, `country`, and `listed_in` contain multiple comma-separated values and need unnesting for deeper analysis.

## 3. Data Cleaning and Preprocessing

In [ ]:
data = df.copy()

data['date_added'] = pd.to_datetime(data['date_added'].str.strip(), errors='coerce')
data['type'] = data['type'].astype('category')
data['rating'] = data['rating'].astype('category')

data['added_year'] = data['date_added'].dt.year
data['added_month'] = data['date_added'].dt.month_name()
data['duration_number'] = data['duration'].str.extract(r'(\d+)').astype(float)
data['duration_unit'] = data['duration'].str.extract(r'([A-Za-z]+)')
data['content_age_when_added'] = data['added_year'] - data['release_year']

data.dtypes

In [ ]:
def split_and_explode(frame, column):
    temp = frame.copy()
    temp[column] = temp[column].fillna('Unknown')
    temp[column] = temp[column].str.split(',')
    temp = temp.explode(column)
    temp[column] = temp[column].str.strip()
    return temp

country_df = split_and_explode(data, 'country')
genre_df = split_and_explode(data, 'listed_in')
actor_df = split_and_explode(data, 'cast')
director_df = split_and_explode(data, 'director')

country_df[['title', 'type', 'country']].head(10)

Unnesting is important because columns like country, cast, director, and genre can contain more than one value in a single row. Without unnesting, a title produced in both India and the United States would be treated as one combined country value instead of two country records.

## 4. Missing Value Detection

In [ ]:
missing_values = df.isna().sum().sort_values(ascending=False)
missing_percent = (missing_values / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'missing_count': missing_values, 'missing_percent': missing_percent})
missing_summary

### Missing Value Observations

- `director` has the highest number of missing values.
- `cast` and `country` also have significant missing values.
- `date_added`, `rating`, and `duration` have very few missing values.
- For business analysis, missing director/cast/country values should be handled carefully because these fields directly affect talent and regional decisions.
- In this notebook, missing values are not deleted. For exploded analysis, missing values are marked as `Unknown` so the original title records are preserved.

## 5. Non-Graphical Analysis

In [ ]:
data['type'].value_counts()

In [ ]:
(data['type'].value_counts(normalize=True) * 100).round(2)

In [ ]:
data['rating'].value_counts().head(10)

In [ ]:
country_df[country_df['country'] != 'Unknown']['country'].value_counts().head(10)

In [ ]:
genre_df['listed_in'].value_counts().head(10)

In [ ]:
actor_df[actor_df['cast'] != 'Unknown']['cast'].value_counts().head(10)

In [ ]:
director_df[director_df['director'] != 'Unknown']['director'].value_counts().head(10)

### Non-Graphical Analysis Observations

- Movies are much more common than TV Shows in the Netflix catalog.
- The United States is the largest country contributor, followed by India and the United Kingdom.
- International Movies, Dramas, Comedies, and International TV Shows are among the strongest content categories.
- Adult and teen ratings such as TV-MA and TV-14 are very common, suggesting Netflix has a strong focus on mature and teen audiences.

## 6. Univariate Visual Analysis

In [ ]:
type_counts = data['type'].value_counts()

plt.figure(figsize=(7, 4))
type_counts.plot(kind='bar', color=['#4c78a8', '#f58518'])
plt.title('Netflix Catalog by Content Type')
plt.xlabel('Content Type')
plt.ylabel('Number of Titles')
plt.xticks(rotation=0)
plt.show()

type_counts

Movies dominate the catalog. This shows that Netflix has historically used movies to build catalog depth, while TV Shows form a smaller but still important share.

In [ ]:
plt.figure(figsize=(9, 4))
data['release_year'].plot(kind='hist', bins=35, color='#54a24b', edgecolor='white')
plt.title('Distribution of Release Year')
plt.xlabel('Release Year')
plt.ylabel('Number of Titles')
plt.show()

data['release_year'].describe()

Most titles are from recent decades, especially after 2000. This means Netflix's catalog is weighted toward modern content, although it also includes older library titles.

In [ ]:
movies = data[data['type'] == 'Movie']
shows = data[data['type'] == 'TV Show']

plt.figure(figsize=(9, 4))
movies['duration_number'].plot(kind='hist', bins=35, color='#e45756', edgecolor='white')
plt.title('Movie Duration Distribution')
plt.xlabel('Duration in Minutes')
plt.ylabel('Number of Movies')
plt.show()

movies['duration_number'].describe()

Movie durations are concentrated around typical feature-film length. The median movie duration is around 98 minutes, so 90-110 minute films are a strong baseline format.

## 7. Bivariate Visual Analysis

In [ ]:
top_countries = country_df[country_df['country'] != 'Unknown']['country'].value_counts().head(10).index
country_plot_data = country_df[country_df['country'].isin(top_countries)].reset_index(drop=True)
country_type_table = pd.crosstab(
    country_plot_data['country'],
    country_plot_data['type']
).reindex(top_countries)

country_type_table.plot(kind='barh', figsize=(10, 5), color=['#4c78a8', '#f58518'])
plt.title('Top Countries by Content Type')
plt.xlabel('Number of Titles')
plt.ylabel('Country')
plt.gca().invert_yaxis()
plt.show()

country_type_table

The United States has the largest catalog presence. India and the United Kingdom are also major contributors. This suggests that Netflix should use different regional content strategies instead of one global strategy.

In [ ]:
added_by_year = data.dropna(subset=['added_year']).groupby(['added_year', 'type'], observed=True).size().unstack(fill_value=0)

added_by_year.plot(kind='line', marker='o', figsize=(10, 5), color=['#4c78a8', '#f58518'])
plt.title('Titles Added to Netflix by Year')
plt.xlabel('Year Added')
plt.ylabel('Number of Titles')
plt.show()

added_by_year.tail(10)

Netflix added content most aggressively in the late 2010s and early 2020s. Movies still outnumber TV Shows in most years, but TV Shows remain a meaningful part of new additions.

In [ ]:
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
tv_month_counts = shows['added_month'].value_counts().reindex(month_order)

plt.figure(figsize=(10, 4))
tv_month_counts.plot(kind='bar', color='#72b7b2')
plt.title('TV Shows Added by Month')
plt.xlabel('Month')
plt.ylabel('Number of TV Shows')
plt.xticks(rotation=35)
plt.show()

tv_month_counts.sort_values(ascending=False)

September has the highest number of TV Show additions in this dataset. This suggests that September can be considered a strong launch month, though Netflix should also test launch timing by region and genre.

In [ ]:
top_ratings = movies['rating'].value_counts().head(8).index
box_data = [movies.loc[movies['rating'] == rating, 'duration_number'].dropna() for rating in top_ratings]

plt.figure(figsize=(10, 5))
plt.boxplot(box_data, tick_labels=top_ratings, patch_artist=True)
plt.title('Movie Duration by Rating')
plt.xlabel('Rating')
plt.ylabel('Duration in Minutes')
plt.show()

movies.groupby('rating', observed=True)['duration_number'].describe().loc[top_ratings]

Movie duration varies by rating. Mature and teen ratings contain many standard-length and longer movies, while children/family-focused ratings tend to be shorter.

In [ ]:
top_genres = genre_df['listed_in'].value_counts().head(10)

plt.figure(figsize=(10, 5))
top_genres.plot(kind='barh', color='#b279a2')
plt.title('Top Genres / Categories')
plt.xlabel('Number of Title-Genre Records')
plt.ylabel('Genre')
plt.gca().invert_yaxis()
plt.show()

top_genres

International Movies, Dramas, and Comedies are among the largest categories. This shows that international and broadly appealing story formats are important parts of Netflix's catalog.

## 8. Outlier Check

In [ ]:
def iqr_outlier_summary(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = series[(series < lower) | (series > upper)]
    return pd.Series({
        'q1': q1,
        'q3': q3,
        'iqr': iqr,
        'lower_bound': lower,
        'upper_bound': upper,
        'outlier_count': outliers.count()
    })

movie_outliers = iqr_outlier_summary(movies['duration_number'].dropna())
tv_outliers = iqr_outlier_summary(shows['duration_number'].dropna())

pd.DataFrame({'movie_duration': movie_outliers, 'tv_show_seasons': tv_outliers})

Outliers are not necessarily incorrect data. Very short movies, very long movies, stand-up specials, documentaries, and long-running TV Shows are valid business categories. Therefore, outlier treatment is optional for this case study.

## 9. Business Insights

1. Movies dominate the Netflix catalog. This means movies are important for catalog breadth and quick content variety.
2. TV Shows are fewer than Movies, but they are important because they can create repeat viewing across episodes and seasons.
3. The United States is the strongest content contributor, but India and the United Kingdom also have major catalog presence.
4. International content is a major pattern in the catalog. International Movies and International TV Shows appear among the top categories.
5. Most movies are around standard feature-film duration, with the median close to 98 minutes.
6. Many TV Shows have only one or two seasons, which suggests limited series and short-season formats are common.
7. Mature and teen ratings such as TV-MA and TV-14 are highly common, showing a strong focus on adult and teen audiences.
8. September has the highest TV Show additions in the dataset, making it a useful month to consider for launches.

## 10. Recommendations

1. Continue producing movies because they form the majority of the catalog and help maintain content variety.
2. Invest more in short-season TV Shows and limited series because they can improve engagement without requiring a long production commitment.
3. Prioritize regional content strategies for the United States, India, the United Kingdom, Canada, France, Japan, South Korea, and Spain.
4. Produce more international dramas, comedies, documentaries, and thrillers because international categories are already strong in the catalog.
5. Use September as a preferred launch window for important TV Shows, while testing whether the same timing works in every country.
6. Use popular regional actors and directors to improve local relevance, but validate their popularity by country before investing large budgets.
7. Maintain a balanced content slate: movies for catalog depth, TV Shows for engagement, and documentaries/limited series for lower-risk experimentation.
8. Improve metadata quality for director, cast, and country because these fields affect recommendations, search, and regional business analysis.

## Final Conclusion

Netflix should continue using movies to maintain a broad catalog, while increasing investment in short-season TV Shows and international originals to drive engagement and regional growth. The strongest expansion opportunities come from combining local country-level production with globally discoverable genres such as drama, comedy, documentary, thriller, and international content.

In [2]:
from scipy.stats import binom

binom.pmf(4, n=10, p=1/4) + binom.pmf(5, n=10, p=1/4) + binom.pmf(6, n=10, p=1/4) + binom.pmf(7, n=10, p=1/4) + binom.pmf(8, n=10, p=1/4) + binom.pmf(9, n=10, p=1/4) + binom.pmf(10, n=10, p=1/4)

np.float64(0.22412490844726554)

In [7]:
1 - binom.cdf(3, n=10, p=1/4)

np.float64(0.22412490844726562)

In [10]:
binom.pmf(k=2, n=10, p=1/4)

np.float64(0.28156757354736334)